# Notebook 03 — Reversible LLM: Max-Batch Search & Full 50M-Token Run

**Series:** Reversible-LLM Training Experiments  
**Goal:** (1) Find the largest batch size that fits in GPU VRAM using a step-up OOM probe,  
then (2) run the full 50 M-token training at that batch size using the **Euler reversible block**  
(winner from Notebook 02), and (3) consolidate all three experiment results into one comparison table.

| Step | Description |
|------|-------------|
| 1 | Install dependencies & check GPU |
| 2 | Define Euler reversible model (same as NB-02 winner) |
| 3 | **Max-batch probe** — step-up loop with OOM catch |
| 4 | Full 50 M-token training at max batch |
| 5 | Load NB-01 / NB-02 metrics & build comparison table |
| 6 | Plot all three loss curves |


In [ ]:
# @title Step 1 — Install dependencies
import subprocess, sys

pkgs = [
    'torch>=2.2.0',
    'datasets',
    'tokenizers',
    'transformers',
    'matplotlib',
    'tqdm',
]

for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed.')


In [ ]:
# @title GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found — please enable a GPU runtime in Colab (Runtime > Change runtime type > T4).')

device = torch.device('cuda')
props  = torch.cuda.get_device_properties(device)
total_vram_gb = props.total_memory / 1024**3

print(f'GPU : {props.name}')
print(f'VRAM: {total_vram_gb:.1f} GB')
print(f'CUDA: {torch.version.cuda}')
print(f'PyTorch: {torch.__version__}')


In [ ]:
# @title Imports & shared config
import math, time, json, os, gc, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Shared hyper-parameters (match NB-01 / NB-02) ────────────────────────────
VOCAB_SIZE   = 8_000
SEQ_LEN      = 256
D_MODEL      = 256
N_HEADS      = 8
N_LAYERS     = 8          # 4 reversible blocks = 8 sub-layers
D_FF         = 1_024
DROPOUT      = 0.0
LR           = 3e-4
WARMUP_STEPS = 200
TARGET_TOKENS = 50_000_000  # 50 M tokens
LOG_EVERY    = 50           # steps

device = torch.device('cuda')
print('Config loaded.')


## Tokenizer

We train a small BPE tokenizer on a slice of WikiText-103 — identical to NB-01 / NB-02 so results are comparable.


In [ ]:
# @title Train BPE tokenizer on WikiText-103
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

print('Loading WikiText-103 ...')
raw = load_dataset('wikitext', 'wikitext-103-raw-v1', split='train')

# Use 20 k articles for tokenizer training (fast)
sample_texts = [t for t in raw['text'][:20_000] if len(t.strip()) > 40]

tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=['[PAD]', '[UNK]', '[BOS]', '[EOS]'],
)
tokenizer.train_from_iterator(sample_texts, trainer=trainer)

PAD_ID = tokenizer.token_to_id('[PAD]')
print(f'Vocab size: {tokenizer.get_vocab_size()} | PAD id: {PAD_ID}')


## Dataset

Tokenise the **full** WikiText-103 training split into fixed-length chunks of `SEQ_LEN = 256` tokens.  
We stop after we have at least `TARGET_TOKENS` tokens so the 50 M run can complete.


In [ ]:
# @title Build token-chunk dataset

class TokenDataset(Dataset):
    def __init__(self, tokens: np.ndarray, seq_len: int):
        self.tokens  = tokens
        self.seq_len = seq_len
        self.n       = (len(tokens) - 1) // seq_len

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        start = idx * self.seq_len
        x = torch.tensor(self.tokens[start     : start + self.seq_len], dtype=torch.long)
        y = torch.tensor(self.tokens[start + 1 : start + self.seq_len + 1], dtype=torch.long)
        return x, y


def build_dataset(raw_dataset, tokenizer, seq_len, target_tokens):
    all_ids = []
    for text in tqdm(raw_dataset['text'], desc='Tokenising'):
        if not text.strip():
            continue
        enc = tokenizer.encode(text)
        all_ids.extend(enc.ids)
        if len(all_ids) >= target_tokens + seq_len:
            break
    tokens = np.array(all_ids, dtype=np.int32)
    print(f'Total tokens tokenised: {len(tokens):,}')
    return TokenDataset(tokens, seq_len)


dataset = build_dataset(raw, tokenizer, SEQ_LEN, TARGET_TOKENS)
print(f'Dataset chunks: {len(dataset):,}  |  '
      f'Effective tokens: {len(dataset) * SEQ_LEN:,}')


## Euler Reversible Transformer

This is the **Euler-step reversible block** from Notebook 02 — the primary (and better) variant.  

### How it works

Standard residual: $y = x + F(x)$  
Euler reversible (Gomez et al. style, two streams):

$$x_1^{\ell+1} = x_1^\ell + \text{Attn}(x_2^\ell)$$
$$x_2^{\ell+1} = x_2^\ell + \text{FFN}(x_1^{\ell+1})$$

**Exact reconstruction** (no stored activations needed during backward):

$$x_2^\ell = x_2^{\ell+1} - \text{FFN}(x_1^{\ell+1})$$
$$x_1^\ell = x_1^{\ell+1} - \text{Attn}(x_2^\ell)$$

This halves activation memory vs a standard Transformer of the same depth.


In [ ]:
# @title Euler Reversible Transformer definition

# ── Attention ─────────────────────────────────────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.scale   = self.d_head ** -0.5
        self.qkv     = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj    = nn.Linear(d_model, d_model, bias=False)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        H, Dh = self.n_heads, self.d_head
        q, k, v = self.qkv(x).split(C, dim=-1)
        q = q.view(B, T, H, Dh).transpose(1, 2)
        k = k.view(B, T, H, Dh).transpose(1, 2)
        v = v.view(B, T, H, Dh).transpose(1, 2)
        # Causal attention via PyTorch's flash-attention path
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                             dropout_p=self.drop.p if self.training else 0.0)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)


# ── Feed-forward ──────────────────────────────────────────────────────────────
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


# ── Euler Reversible Block ────────────────────────────────────────────────────
class EulerReversibleBlock(nn.Module):
    """One reversible layer: two streams (x1, x2).

    Forward:
        x1_out = x1 + Attn(LayerNorm(x2))
        x2_out = x2 + FFN(x1_out)

    The backward pass can reconstruct x1, x2 from x1_out, x2_out
    without storing intermediate activations.
    """

    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff   = FeedForward(d_model, d_ff, dropout)

    def forward(self, x1, x2):
        x1_out = x1 + self.attn(self.ln1(x2))
        x2_out = x2 + self.ff(x1_out)
        return x1_out, x2_out


# ── Full reversible model ─────────────────────────────────────────────────────
class ReversibleTransformer(nn.Module):
    """Stack of EulerReversibleBlocks with shared embedding.

    The two streams x1 / x2 are both initialised from the token embedding
    and averaged before the final LM head.
    """

    def __init__(self, vocab_size, d_model, n_heads, n_blocks, d_ff,
                 seq_len, dropout=0.0):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList([
            EulerReversibleBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_blocks)
        ])
        self.ln_f    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)
        # Weight tying
        self.head.weight = self.tok_emb.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        emb  = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        # Both streams start from the same embedding
        x1, x2 = emb, emb.clone()
        for block in self.blocks:
            x1, x2 = block(x1, x2)
        # Merge streams
        out = self.ln_f((x1 + x2) * 0.5)
        return self.head(out)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())


# ── Sanity check ──────────────────────────────────────────────────────────────
_m = ReversibleTransformer(
    VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS // 2, D_FF, SEQ_LEN, DROPOUT
).to(device)
_x = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=device)
_ = _m(_x)
print(f'Model params: {_m.num_params():,}')
del _m, _x
torch.cuda.empty_cache()


In [ ]:
# @title Learning-rate scheduler (cosine with linear warm-up)

def get_lr(step, lr, warmup_steps, total_steps):
    """Linear warm-up then cosine decay to lr/10."""
    if step < warmup_steps:
        return lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return lr / 10 + (lr - lr / 10) * 0.5 * (1 + math.cos(math.pi * progress))

print('Scheduler defined.')


## Step 3 — Max-Batch Probe

### Why do this?

Larger batches → more GPU utilisation → higher tokens/sec → shorter wall-clock training time.  
But exceeding VRAM capacity causes an **Out-Of-Memory (OOM) RuntimeError**.

### Algorithm

```
candidates = [32, 64, 96, 128, 192, 256, 320, 384]
last_ok = None

for B in candidates:
    try:
        run 5 forward+backward steps at batch size B
        last_ok = B          # succeeded
    except RuntimeError (OOM):
        break                # too large

max_batch = last_ok
```

Between every attempt we call `torch.cuda.empty_cache()` and  
`torch.cuda.reset_peak_memory_stats()` to get clean measurements.

> **Note:** The probe creates a fresh model instance and optimizer each time so  
> residual optimizer state from previous attempts cannot inflate memory readings.


In [ ]:
# @title Max-batch probe implementation

def probe_batch_size(
    batch_candidates,
    vocab_size, d_model, n_heads, n_blocks, d_ff, seq_len, dropout,
    probe_steps=5,
    device=None,
):
    """Step-up OOM probe.

    For each candidate batch size, instantiate a fresh model+optimizer and run
    `probe_steps` forward+backward passes.  Records peak VRAM and throughput
    for each successful size.  Returns (max_batch, results_list).

    Parameters
    ----------
    batch_candidates : list[int]
        Ordered (ascending) batch sizes to probe.
    probe_steps : int
        How many steps to run per candidate (more = more stable measurement).

    Returns
    -------
    max_batch : int
    results   : list[dict]
    """
    if device is None:
        device = torch.device('cuda')

    results  = []
    max_batch = None
    criterion = nn.CrossEntropyLoss()

    print('=' * 60)
    print(f'  Batch-size probe  |  probe_steps={probe_steps}')
    print('=' * 60)
    print(f'{"Batch":>8}  {"Status":>10}  {"Peak VRAM (GB)":>16}  {"Tok/s":>10}')
    print('-' * 60)

    for batch in batch_candidates:
        # ── clean slate ──────────────────────────────────────────────────────
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

        try:
            # fresh model + optimizer
            model = ReversibleTransformer(
                vocab_size, d_model, n_heads, n_blocks, d_ff, seq_len, dropout
            ).to(device)
            opt = torch.optim.AdamW(model.parameters(), lr=LR)
            model.train()

            t0 = time.perf_counter()
            for _ in range(probe_steps):
                x = torch.randint(0, vocab_size, (batch, seq_len), device=device)
                y = torch.randint(0, vocab_size, (batch, seq_len), device=device)
                logits = model(x)
                loss   = criterion(logits.view(-1, vocab_size), y.view(-1))
                loss.backward()
                opt.step()
                opt.zero_grad(set_to_none=True)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0

            peak_gb = torch.cuda.max_memory_allocated(device) / 1024**3
            toks_s  = (probe_steps * batch * seq_len) / elapsed

            print(f'{batch:>8}  {"OK":>10}  {peak_gb:>14.2f}  {toks_s:>10,.0f}')
            results.append(dict(batch=batch, status='ok',
                               peak_gb=peak_gb, toks_s=toks_s))
            max_batch = batch

        except RuntimeError as e:
            if 'out of memory' in str(e).lower() or 'CUDA out of memory' in str(e):
                print(f'{batch:>8}  {"OOM":>10}  {"—":>16}  {"—":>10}')
                results.append(dict(batch=batch, status='oom',
                                   peak_gb=None, toks_s=None))
            else:
                raise
        finally:
            # always free resources
            try:
                del model, opt
            except NameError:
                pass
            gc.collect()
            torch.cuda.empty_cache()

    print('=' * 60)
    if max_batch is None:
        raise RuntimeError('All batch sizes caused OOM — even batch=32 fails!')
    print(f'\n>>> Max successful batch size: {max_batch}')
    return max_batch, results


print('Probe function defined.')


In [ ]:
# @title Run the max-batch probe

BATCH_CANDIDATES = [32, 64, 96, 128, 192, 256, 320, 384]

MAX_BATCH, probe_results = probe_batch_size(
    batch_candidates = BATCH_CANDIDATES,
    vocab_size  = VOCAB_SIZE,
    d_model     = D_MODEL,
    n_heads     = N_HEADS,
    n_blocks    = N_LAYERS // 2,
    d_ff        = D_FF,
    seq_len     = SEQ_LEN,
    dropout     = DROPOUT,
    probe_steps = 5,
)

print(f'\nUsing MAX_BATCH = {MAX_BATCH} for the full training run.')


In [ ]:
# @title Display probe results as a formatted table

print(f'\n{"Batch":>8}  {"Status":>8}  {"Peak VRAM (GB)":>16}  {"Tokens/sec":>12}')
print('-' * 52)
for r in probe_results:
    gb  = f"{r['peak_gb']:.2f}" if r['peak_gb'] is not None else '—'
    tks = f"{r['toks_s']:,.0f}" if r['toks_s'] is not None else '—'
    print(f"{r['batch']:>8}  {r['status']:>8}  {gb:>16}  {tks:>12}")

print(f'\n>>> Selected batch size for full run: MAX_BATCH = {MAX_BATCH}')


## Gradient Accumulation (optional)

If `MAX_BATCH < 128` the effective batch can be raised via gradient accumulation:

```
effective_batch = micro_batch × grad_accum_steps
```

Each micro-step runs a forward+backward but only calls `optimizer.step()` every
`grad_accum_steps` micro-steps.  This is **mathematically equivalent** to training
with `effective_batch` examples at once (for SGD / AdamW with mean-reduced loss).

In the cell below, `GRAD_ACCUM` is set so that `effective_batch ≈ 128` regardless
of what `MAX_BATCH` the probe returned, unless `MAX_BATCH ≥ 128` already.


In [ ]:
# @title Set micro_batch and grad_accum for the full run

TARGET_EFFECTIVE_BATCH = 128  # desired effective batch size

MICRO_BATCH  = MAX_BATCH
GRAD_ACCUM   = max(1, TARGET_EFFECTIVE_BATCH // MICRO_BATCH)
EFF_BATCH    = MICRO_BATCH * GRAD_ACCUM

TOTAL_STEPS  = math.ceil(TARGET_TOKENS / (EFF_BATCH * SEQ_LEN))

print(f'MICRO_BATCH       = {MICRO_BATCH}')
print(f'GRAD_ACCUM steps  = {GRAD_ACCUM}')
print(f'EFFECTIVE BATCH   = {EFF_BATCH}')
print(f'SEQ_LEN           = {SEQ_LEN}')
print(f'Tokens per step   = {EFF_BATCH * SEQ_LEN:,}')
print(f'Total steps       = {TOTAL_STEPS:,}')
print(f'Target tokens     = {TARGET_TOKENS:,}')


## Step 4 — Full 50 M-Token Training Run

We now train the Euler reversible model for the full 50 M tokens at `MAX_BATCH` and record:

- **Loss curve** (logged every `LOG_EVERY` optimiser steps)
- **Tokens/sec** (average over the whole run)
- **Peak VRAM** (via `torch.cuda.max_memory_allocated`)
- **Wall-clock time**


In [ ]:
# @title Full 50 M-token training

# ── Build DataLoader ─────────────────────────────────────────────────────────
loader = DataLoader(
    dataset,
    batch_size  = MICRO_BATCH,
    shuffle     = True,
    drop_last   = True,
    num_workers = 2,
    pin_memory  = True,
)

# ── Build model & optimizer ───────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)

model = ReversibleTransformer(
    VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS // 2, D_FF, SEQ_LEN, DROPOUT
).to(device)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1)
crit  = nn.CrossEntropyLoss()

n_params = model.num_params()
print(f'Model params: {n_params:,}')

# ── Training loop ─────────────────────────────────────────────────────────────
loss_log     = []   # (step, loss)
tokens_seen  = 0
opt_step     = 0
accum_loss   = 0.0
accum_count  = 0

t_start = time.perf_counter()
model.train()

pbar = tqdm(total=TOTAL_STEPS, desc='Training', dynamic_ncols=True)
data_iter = iter(loader)

micro_step = 0

while opt_step < TOTAL_STEPS:
    # ── fetch next micro-batch (re-cycle loader) ──────────────────────────────
    try:
        x, y = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        x, y = next(data_iter)

    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

    # ── forward + backward ────────────────────────────────────────────────────
    logits = model(x)
    loss   = crit(logits.view(-1, VOCAB_SIZE), y.view(-1)) / GRAD_ACCUM
    loss.backward()

    accum_loss  += loss.item() * GRAD_ACCUM
    accum_count += 1
    tokens_seen += MICRO_BATCH * SEQ_LEN
    micro_step  += 1

    # ── optimiser step every GRAD_ACCUM micro-steps ───────────────────────────
    if micro_step % GRAD_ACCUM == 0:
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        # LR schedule
        lr_now = get_lr(opt_step, LR, WARMUP_STEPS, TOTAL_STEPS)
        for pg in opt.param_groups:
            pg['lr'] = lr_now

        opt.step()
        opt.zero_grad(set_to_none=True)
        opt_step += 1

        avg_loss = accum_loss / accum_count
        accum_loss, accum_count = 0.0, 0

        if opt_step % LOG_EVERY == 0 or opt_step == 1:
            elapsed   = time.perf_counter() - t_start
            toks_s    = tokens_seen / elapsed
            peak_vram = torch.cuda.max_memory_allocated(device) / 1024**3
            loss_log.append((opt_step, avg_loss))
            pbar.set_postfix(loss=f'{avg_loss:.4f}',
                             tok_s=f'{toks_s:,.0f}',
                             vram=f'{peak_vram:.1f}GB')

        pbar.update(1)

pbar.close()
torch.cuda.synchronize()

# ── Final metrics ─────────────────────────────────────────────────────────────
t_total    = time.perf_counter() - t_start
final_loss = loss_log[-1][1] if loss_log else float('nan')
avg_toks_s = tokens_seen / t_total
peak_gb    = torch.cuda.max_memory_allocated(device) / 1024**3

print('\n' + '=' * 55)
print('  Max-Batch Reversible Training Complete')
print('=' * 55)
print(f'  Params        : {n_params:,}')
print(f'  Micro batch   : {MICRO_BATCH}')
print(f'  Grad accum    : {GRAD_ACCUM}')
print(f'  Eff. batch    : {EFF_BATCH}')
print(f'  Seq len       : {SEQ_LEN}')
print(f'  Tokens seen   : {tokens_seen:,}')
print(f'  Opt steps     : {opt_step:,}')
print(f'  Final loss    : {final_loss:.4f}')
print(f'  Avg tok/s     : {avg_toks_s:,.0f}')
print(f'  Peak VRAM     : {peak_gb:.2f} GB')
print(f'  Wall time     : {t_total/60:.1f} min  ({t_total:.1f} s)')
print('=' * 55)


## Step 5 — Save Metrics & Build Final Comparison Table

We save the max-batch run metrics, then load (or stub) metrics from NB-01 and NB-02  
to build a unified comparison.


In [ ]:
# @title Save max-batch metrics

os.makedirs('results/metrics', exist_ok=True)

maxbatch_metrics = {
    'experiment'   : 'Reversible-MaxBatch',
    'model'        : 'EulerReversible',
    'n_params'     : n_params,
    'micro_batch'  : MICRO_BATCH,
    'grad_accum'   : GRAD_ACCUM,
    'eff_batch'    : EFF_BATCH,
    'seq_len'      : SEQ_LEN,
    'tokens_seen'  : tokens_seen,
    'opt_steps'    : opt_step,
    'final_loss'   : final_loss,
    'avg_toks_s'   : avg_toks_s,
    'peak_vram_gb' : peak_gb,
    'wall_time_s'  : t_total,
    'loss_curve'   : loss_log,
    'probe_results': probe_results,
    'max_batch_found': MAX_BATCH,
}

with open('results/metrics/nb03_maxbatch_metrics.json', 'w') as f:
    json.dump(maxbatch_metrics, f, indent=2)

print('Saved: results/metrics/nb03_maxbatch_metrics.json')


In [ ]:
# @title Load or stub NB-01 (Baseline) and NB-02 (Reversible-Euler) metrics
#
# If you have run notebooks 01 and 02 and their metrics JSON files exist,
# they will be loaded automatically.  Otherwise, representative stub values
# are used so the comparison table can still render.

NB01_PATH = 'results/metrics/nb01_baseline_metrics.json'
NB02_PATH = 'results/metrics/nb02_euler_metrics.json'

# ── Baseline (NB-01) ──────────────────────────────────────────────────────────
if os.path.exists(NB01_PATH):
    with open(NB01_PATH) as f:
        nb01 = json.load(f)
    print(f'Loaded NB-01 metrics from {NB01_PATH}')
else:
    print(f'[STUB] {NB01_PATH} not found — using illustrative values.')
    nb01 = {
        'experiment'   : 'Baseline',
        'model'        : 'StandardTransformer',
        'n_params'     : 20_300_000,
        'micro_batch'  : 32,
        'grad_accum'   : 1,
        'eff_batch'    : 32,
        'seq_len'      : 256,
        'tokens_seen'  : 50_000_000,
        'final_loss'   : 4.21,
        'avg_toks_s'   : 95_000,
        'peak_vram_gb' : 4.8,
        'wall_time_s'  : 530.0,
        'loss_curve'   : [(50*i, 6.5 - i*0.08) for i in range(1, 14)],
    }

# ── Reversible-Euler NB-02 ────────────────────────────────────────────────────
if os.path.exists(NB02_PATH):
    with open(NB02_PATH) as f:
        nb02 = json.load(f)
    print(f'Loaded NB-02 metrics from {NB02_PATH}')
else:
    print(f'[STUB] {NB02_PATH} not found — using illustrative values.')
    nb02 = {
        'experiment'   : 'Reversible-Euler',
        'model'        : 'EulerReversible',
        'n_params'     : 21_500_000,
        'micro_batch'  : 32,
        'grad_accum'   : 1,
        'eff_batch'    : 32,
        'seq_len'      : 256,
        'tokens_seen'  : 50_000_000,
        'final_loss'   : 4.17,
        'avg_toks_s'   : 88_000,
        'peak_vram_gb' : 3.9,
        'wall_time_s'  : 570.0,
        'loss_curve'   : [(50*i, 6.4 - i*0.083) for i in range(1, 14)],
    }

nb03 = maxbatch_metrics
nb03['experiment'] = 'Reversible-MaxBatch'

print('All three experiment metrics ready.')


In [ ]:
# @title Build & print the final comparison table

exps = [nb01, nb02, nb03]

def fmt_batch(e):
    if e.get('grad_accum', 1) > 1:
        return f"{e['micro_batch']}×{e['grad_accum']}={e['eff_batch']}"
    return str(e['eff_batch'])

print('\n' + '=' * 110)
print('  FINAL EXPERIMENT COMPARISON')
print('=' * 110)

hdr = (f'{"Experiment":<24}  {"Params":>10}  {"Batch":>14}  '
       f'{"SeqLen":>7}  {"Tokens":>12}  {"FinalLoss":>10}  '
       f'{"Tok/s":>10}  {"PeakVRAM":>10}  {"Time(min)":>10}')
print(hdr)
print('-' * 110)

for e in exps:
    print(
        f"{e['experiment']:<24}  "
        f"{e['n_params']:>10,}  "
        f"{fmt_batch(e):>14}  "
        f"{e['seq_len']:>7}  "
        f"{e['tokens_seen']:>12,}  "
        f"{e['final_loss']:>10.4f}  "
        f"{e['avg_toks_s']:>10,.0f}  "
        f"{e['peak_vram_gb']:>10.2f}  "
        f"{e['wall_time_s']/60:>10.1f}"
    )

print('=' * 110)
print()

# ── Delta analysis ────────────────────────────────────────────────────────────
loss_delta_vs_base   = nb03['final_loss'] - nb01['final_loss']
loss_delta_vs_euler  = nb03['final_loss'] - nb02['final_loss']
toks_speedup         = nb03['avg_toks_s'] / nb01['avg_toks_s']
vram_delta           = nb03['peak_vram_gb'] - nb01['peak_vram_gb']

print('Delta analysis (MaxBatch vs Baseline):')
print(f'  Loss difference   : {loss_delta_vs_base:+.4f}')
print(f'  Tokens/s ratio    : {toks_speedup:.2f}x')
print(f'  Peak VRAM delta   : {vram_delta:+.2f} GB')
print(f'  Time ratio        : {nb03["wall_time_s"]/nb01["wall_time_s"]:.2f}x')
print()
print('Delta analysis (MaxBatch vs Reversible-Euler):')
print(f'  Loss difference   : {loss_delta_vs_euler:+.4f}')
print(f'  Tokens/s ratio    : {nb03["avg_toks_s"]/nb02["avg_toks_s"]:.2f}x')


In [ ]:
# @title Save final_comparison.json

os.makedirs('results/metrics', exist_ok=True)

comparison = {
    'description': (
        'Final comparison of all three reversible-LLM experiments. '
        'Generated by notebook 03_max_batch_reversible.ipynb.'
    ),
    'experiments': [
        {
            'experiment'   : e['experiment'],
            'model'        : e.get('model', 'Unknown'),
            'n_params'     : e['n_params'],
            'micro_batch'  : e.get('micro_batch', e.get('eff_batch', 32)),
            'grad_accum'   : e.get('grad_accum', 1),
            'eff_batch'    : e.get('eff_batch', e.get('micro_batch', 32)),
            'seq_len'      : e['seq_len'],
            'tokens_seen'  : e['tokens_seen'],
            'final_loss'   : e['final_loss'],
            'avg_toks_s'   : e['avg_toks_s'],
            'peak_vram_gb' : e['peak_vram_gb'],
            'wall_time_s'  : e['wall_time_s'],
        }
        for e in exps
    ],
    'max_batch_probe': {
        'max_batch_found': MAX_BATCH,
        'probe_results'  : probe_results,
    },
}

out_path = 'results/metrics/final_comparison.json'
with open(out_path, 'w') as f:
    json.dump(comparison, f, indent=2)

print(f'Saved: {out_path}')


## Step 6 — Loss Curve Comparison

All three experiments on one chart.  
The x-axis shows optimiser steps (each step processes `eff_batch × seq_len` tokens).


In [ ]:
# @title Plot all three loss curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4e79a7', '#f28e2b', '#e15759']
labels = ['Baseline (NB-01)', 'Reversible-Euler (NB-02)', 'Reversible-MaxBatch (NB-03)']

for ax_idx, ax in enumerate(axes):
    for exp, color, label in zip(exps, colors, labels):
        curve = exp.get('loss_curve', [])
        if not curve:
            continue
        steps  = [s for s, _ in curve]
        losses = [l for _, l in curve]
        ax.plot(steps, losses, color=color, label=label,
                linewidth=2, alpha=0.9, marker='o' if len(steps) < 30 else None,
                markersize=4)

    ax.set_xlabel('Optimiser Step', fontsize=11)
    ax.set_ylabel('Cross-Entropy Loss', fontsize=11)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_title('Training Loss — All Experiments', fontsize=12, fontweight='bold')

# Second panel: zoom into the last 40% of steps
all_steps = [s for e in exps for s, _ in e.get('loss_curve', [])]
if all_steps:
    cutoff = sorted(all_steps)[int(0.6 * len(all_steps))]
    axes[1].set_xlim(left=cutoff)
axes[1].set_title('Training Loss — Late Training (Zoom)', fontsize=12, fontweight='bold')

fig.suptitle('Reversible LLM Experiments — Loss Comparison', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()

os.makedirs('results/figures', exist_ok=True)
fig.savefig('results/figures/loss_comparison_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to results/figures/loss_comparison_all.png')


In [ ]:
# @title VRAM & throughput bar charts

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(13, 4))

exp_names  = [e['experiment'].replace('-', '-\n') for e in exps]
bar_colors = ['#4e79a7', '#f28e2b', '#e15759']

# Peak VRAM
vrm = [e['peak_vram_gb'] for e in exps]
bars = ax1.bar(exp_names, vrm, color=bar_colors, width=0.5, edgecolor='white')
ax1.bar_label(bars, fmt='%.2f GB', padding=3, fontsize=9)
ax1.set_ylabel('Peak VRAM (GB)')
ax1.set_title('Peak VRAM Usage')
ax1.set_ylim(0, max(vrm) * 1.3)

# Tokens/sec
tks = [e['avg_toks_s'] / 1000 for e in exps]
bars2 = ax2.bar(exp_names, tks, color=bar_colors, width=0.5, edgecolor='white')
ax2.bar_label(bars2, fmt='%.1f K', padding=3, fontsize=9)
ax2.set_ylabel('Tokens / second (K)')
ax2.set_title('Training Throughput')
ax2.set_ylim(0, max(tks) * 1.3)

# Final loss
fl = [e['final_loss'] for e in exps]
bars3 = ax3.bar(exp_names, fl, color=bar_colors, width=0.5, edgecolor='white')
ax3.bar_label(bars3, fmt='%.4f', padding=3, fontsize=9)
ax3.set_ylabel('Final Cross-Entropy Loss')
ax3.set_title('Final Loss')
ax3.set_ylim(min(fl) * 0.97, max(fl) * 1.02)

for ax in (ax1, ax2, ax3):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')

fig.suptitle('Experiment Summary — VRAM, Throughput, Final Loss',
             fontsize=12, fontweight='bold')
plt.tight_layout()
fig.savefig('results/figures/summary_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Bar chart saved to results/figures/summary_bars.png')


In [ ]:
# @title Max-batch probe visualisation

ok_results  = [r for r in probe_results if r['status'] == 'ok']
oom_batches = [r['batch'] for r in probe_results if r['status'] == 'oom']

if ok_results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    batches = [r['batch'] for r in ok_results]
    peak_gb = [r['peak_gb'] for r in ok_results]
    toks_s  = [r['toks_s'] / 1000 for r in ok_results]

    # VRAM vs batch
    ax1.plot(batches, peak_gb, 'o-', color='#e15759', linewidth=2, markersize=7)
    for b in oom_batches:
        ax1.axvline(b, color='red', linestyle='--', alpha=0.5, label=f'OOM @ {b}')
    ax1.axvline(MAX_BATCH, color='green', linestyle=':', linewidth=2,
                label=f'Max batch = {MAX_BATCH}')
    ax1.set_xlabel('Batch size')
    ax1.set_ylabel('Peak VRAM (GB)')
    ax1.set_title('Peak VRAM vs Batch Size')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Throughput vs batch
    ax2.plot(batches, toks_s, 's-', color='#4e79a7', linewidth=2, markersize=7)
    ax2.axvline(MAX_BATCH, color='green', linestyle=':', linewidth=2,
                label=f'Max batch = {MAX_BATCH}')
    ax2.set_xlabel('Batch size')
    ax2.set_ylabel('Tokens / second (K)')
    ax2.set_title('Throughput vs Batch Size')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    for ax in (ax1, ax2):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    fig.suptitle('Max-Batch Probe Results', fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig.savefig('results/figures/probe_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Probe chart saved to results/figures/probe_results.png')
else:
    print('No successful probe results to visualise.')


## Summary & Key Takeaways

### Max-Batch Probe

The step-up probe systematically tries increasing batch sizes, catching `RuntimeError` (CUDA OOM)  
and recording the last successful size.  This gives us the **maximum batch that fits in VRAM**  
without trial-and-error guessing.

### Reversible Blocks — Memory Advantage

Because the Euler reversible block can **reconstruct activations during the backward pass**,  
it stores far fewer intermediate tensors than a standard Transformer.  This is why the reversible  
model can support a larger batch size at the same VRAM budget.

### Gradient Accumulation

When the hardware max batch is smaller than the desired *effective* batch size,  
gradient accumulation bridges the gap with zero mathematical difference:

$$\text{effective\_batch} = \text{micro\_batch} \times \text{grad\_accum\_steps}$$

### Files Written

| File | Description |
|------|-------------|
| `results/metrics/nb03_maxbatch_metrics.json` | Full metrics + probe results for NB-03 |
| `results/metrics/final_comparison.json` | Consolidated 3-experiment comparison |
| `results/figures/loss_comparison_all.png` | Loss curves (all experiments) |
| `results/figures/summary_bars.png` | VRAM / throughput / loss bar charts |
| `results/figures/probe_results.png` | Max-batch probe visualisation |
